# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [3]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [12]:
# 1
print("Строк", clean.sum())

rev_sep = amount[clean & (month == "2026-09")].sum()
rev_oct = amount[clean & (month == "2026-10")].sum()
change = (rev_oct - rev_sep) / rev_sep * 100

print("Сентябрь", rev_sep)
print("Октябрь", rev_oct)
print("Изменение", round(change, 1))

Строк 4739
Сентябрь 35243410.0
Октябрь 34419520.0
Изменение -2.3


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [9]:
# 2
months = np.unique(month)
cats = np.unique(cat)

table_cat = np.array([[amount[clean & (cat == c) & (month == m)].sum() for m in months]
                      for c in cats])
print(cats)
print(table_cat)

total = table_cat.sum(axis=1)         
print("Больше всего выручки", cats[total.argmax()], total.max())

change_cat = (table_cat[:, 1] - table_cat[:, 0]) / table_cat[:, 0] * 100
print("Изменение", np.round(change_cat, 1))
print("Сильнее всех упала", cats[change_cat.argmin()], round(change_cat.min(), 1))

['Дом' 'Книги' 'Красота' 'Одежда' 'Электроника']
[[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]
Больше всего выручки Электроника 34370430.0
Изменение [ 2.6  7.1  2.6 -7.2 -2.5]
Сильнее всех упала Одежда -7.2


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [10]:
# 3
count_cat = np.array([[(clean & (cat == c) & (month == m)).sum() for m in months]
                      for c in cats])
print(count_cat)

avg_cat = table_cat / count_cat      
print(np.round(avg_cat))

print("Самый высокий чек в октябре", cats[avg_cat[:, 1].argmax()], round(avg_cat[:, 1].max()))

[[476 476]
 [265 299]
 [422 435]
 [678 652]
 [531 505]]
[[10499. 10768.]
 [ 4020.  3816.]
 [ 6187.  6159.]
 [13521. 13046.]
 [32773. 33600.]]
Самый высокий чек в октябре Электроника 33600


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [17]:
# 4
cities = np.unique(city)
table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])

share = table / table.sum(axis=0)
print(cities)
print(np.round(share, 3))
print("Сумма по столбцам", share.sum(axis=0))
print("Шымкент", np.round(share[cities == "Шымкент"], 3))

['Актобе' 'Алматы' 'Астана' 'Атырау' 'Караганда' 'Шымкент']
[[0.073 0.074]
 [0.411 0.425]
 [0.256 0.257]
 [0.026 0.028]
 [0.104 0.098]
 [0.129 0.117]]
Сумма по столбцам [1. 1.]
Шымкент [[0.129 0.117]]


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [16]:
# 5
change_city = table[:, 1] - table[:, 0]
contrib = change_city / change_city.sum()

print("Общее изменение", change_city.sum())
print(cities)
print("Вклад", np.round(contrib * 100, 1))
print("Шымкент", np.round(contrib[cities == "Шымкент"] * 100, 1))

Общее изменение -823890.0
['Актобе' 'Алматы' 'Астана' 'Атырау' 'Караганда' 'Шымкент']
Вклад [  4.4 -17.2  22.3  -5.6  33.6  62.5]
Шымкент [62.5]


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [13]:
# 6
share_city = table / table.sum(axis=1, keepdims=True)

print(np.round(share_city, 3))
print("Сумма по строкам", share_city.sum(axis=1))

[[0.504 0.496]
 [0.498 0.502]
 [0.505 0.495]
 [0.488 0.512]
 [0.52  0.48 ]
 [0.53  0.47 ]]
Сумма по строкам [1. 1. 1. 1. 1. 1.]


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [14]:
# 7
x = amount[clean]
z = (x - x.mean()) / x.std()

print("Заказов", (np.abs(z) > 3).sum())
print("Доля", round((np.abs(z) > 3).mean() * 100, 1))
print("Среднее", round(z.mean(), 3), "| std z:", round(z.std(), 3))

Заказов 87
Доля 1.8
Среднее 0.0 | std z: 1.0


## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [19]:
# 8
ok = clean & ~np.isnan(days)
X = np.column_stack([amount[ok], days[ok]])

X_scaled = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))

print(X.shape)
print("min", X_scaled.min(axis=0))
print("max", X_scaled.max(axis=0))

(4595, 2)
min [0. 0.]
max [1. 1.]
